# Pulso — Fase 2: Classificador de Risco com TF-IDF e Regressão Logística

**Projeto:** Pulso — Inteligência Cardiológica com Consciência de Dados  
**Autor:** Richard Schmitz | RM 567951  
**Disciplina:** Inteligência Artificial — FIAP | Turma 2TIAOR-2026  

---

## Objetivo

Este notebook implementa a Parte 2 da Fase 2 do projeto Pulso.  
A partir de frases médicas rotuladas, o sistema:

1. Carrega o dataset `frases_risco.csv` com frases e rótulos de risco
2. Transforma as frases em vetores numéricos com TF-IDF
3. Treina um classificador de Regressão Logística
4. Avalia o desempenho geral do modelo
5. Analisa se o modelo performa de forma diferente para frases com perfil feminino atípico vs. masculino típico

Essa última etapa é a contribuição original do projeto Pulso:  
verificar se o viés de gênero documentado na Fase 1 se manifesta também no classificador.

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas com sucesso.')

## 2. Carregamento e inspeção do dataset

In [ ]:
df = pd.read_csv('../data/frases_risco.csv')

print(f'Total de registros: {len(df)}')
print(f'\nDistribuição por nível de risco:')
print(df['nivel_risco'].value_counts())
print(f'\nDistribuição por perfil de gênero:')
print(df['perfil_genero'].value_counts())
print()
df.head(10)

## 3. Pré-processamento

O TF-IDF (Term Frequency–Inverse Document Frequency) transforma cada frase  
em um vetor numérico onde cada posição representa uma palavra do vocabulário.  
Palavras que aparecem muito em um documento mas pouco no corpus recebem peso maior.

In [ ]:
X = df['frase']
y = df['nivel_risco']

# Divisão treino/teste — estratificada para manter proporção de classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Vetorização TF-IDF
# ngram_range=(1,2) captura bigramas como "dor no" e "falta de"
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'Tamanho do vocabulário TF-IDF: {len(vectorizer.vocabulary_)} termos')
print(f'Amostras de treino : {X_train_tfidf.shape[0]}')
print(f'Amostras de teste  : {X_test_tfidf.shape[0]}')

## 4. Treinamento do classificador

In [ ]:
modelo = LogisticRegression(random_state=42, max_iter=1000)
modelo.fit(X_train_tfidf, y_train)

print('Modelo treinado com sucesso.')

## 5. Avaliação geral do modelo

In [ ]:
y_pred = modelo.predict(X_test_tfidf)

acuracia = accuracy_score(y_test, y_pred)
print(f'Acurácia geral: {acuracia:.2%}\n')

print('Relatório de classificação:')
print(classification_report(y_test, y_pred))

print('Matriz de confusão:')
cm = confusion_matrix(y_test, y_pred, labels=['alto risco', 'baixo risco'])
cm_df = pd.DataFrame(
    cm,
    index=['Real: alto risco', 'Real: baixo risco'],
    columns=['Previsto: alto risco', 'Previsto: baixo risco']
)
print(cm_df)

## 6. Análise de viés de gênero no classificador

Esta é a contribuição original do projeto Pulso nesta fase.  

A hipótese, baseada no eixo temático da Fase 1, é que o modelo pode classificar  
frases com sintomas atípicos (perfil feminino) com menor precisão do que frases  
com sintomas típicos (perfil masculino), porque o vocabulário de alto risco no  
dataset é dominado por termos masculinos clássicos como "dor no peito" e "aperto no tórax".

In [ ]:
# Aplicar o modelo a todo o dataset para análise por subgrupo
X_all_tfidf = vectorizer.transform(df['frase'])
df['predicao'] = modelo.predict(X_all_tfidf)
df['correto'] = (df['predicao'] == df['nivel_risco']).astype(int)

# Análise por perfil de gênero
analise_vies = df.groupby('perfil_genero').agg(
    total=('correto', 'count'),
    acertos=('correto', 'sum')
).reset_index()
analise_vies['acuracia'] = (analise_vies['acertos'] / analise_vies['total'] * 100).round(1)

print('Acurácia do modelo por perfil de gênero:\n')
print(analise_vies.to_string(index=False))

# Análise de falsos negativos em alto risco por perfil
alto_risco = df[df['nivel_risco'] == 'alto risco'].copy()
fn_analise = alto_risco.groupby('perfil_genero').agg(
    total_alto_risco=('correto', 'count'),
    classificados_corretamente=('correto', 'sum')
).reset_index()
fn_analise['falsos_negativos'] = (
    fn_analise['total_alto_risco'] - fn_analise['classificados_corretamente']
)
fn_analise['taxa_falso_negativo'] = (
    fn_analise['falsos_negativos'] / fn_analise['total_alto_risco'] * 100
).round(1)

print('\nFalsos negativos em casos de alto risco por perfil de gênero:')
print('(Falso negativo = paciente de alto risco classificado como baixo risco)\n')
print(fn_analise.to_string(index=False))

print('\nInterpretação:')
print('Uma taxa de falso negativo maior no perfil feminino atípico indica que')
print('o modelo subestima o risco de pacientes com sintomas não clássicos.')
print('Esse padrão reproduz, em escala reduzida, o viés documentado na literatura clínica.')

## 7. Teste com novas frases

Demonstração do modelo em produção com frases não vistas durante o treinamento.

In [ ]:
novas_frases = [
    # Sintoma típico masculino — esperado: alto risco
    "sinto dor forte no peito e suor frio há uma hora",
    # Sintoma atípico feminino — esperado: alto risco, mas pode ser subclassificado
    "estou com cansaço que não passa e uma sensação estranha no estômago",
    # Baixo risco claro
    "leve dor de cabeça após dia estressante que passou com repouso",
    # Sintoma atípico feminino grave
    "tenho sentido náusea e dor no maxilar que aparece e some sem motivo"
]

novas_tfidf = vectorizer.transform(novas_frases)
predicoes = modelo.predict(novas_tfidf)
probabilidades = modelo.predict_proba(novas_tfidf)
classes = modelo.classes_

print('Classificação de novas frases:\n')
for frase, pred, prob in zip(novas_frases, predicoes, probabilidades):
    idx_alto = list(classes).index('alto risco')
    confianca = prob[idx_alto]
    print(f'Frase     : {frase}')
    print(f'Resultado : {pred} (confiança alto risco: {confianca:.1%})')
    print()

## 8. Termos mais relevantes por classe

Quais palavras o modelo associa mais fortemente a cada nível de risco.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coeficientes = modelo.coef_[0]

# Índices dos maiores coeficientes positivos (alto risco) e negativos (baixo risco)
top_alto  = np.argsort(coeficientes)[-10:][::-1]
top_baixo = np.argsort(coeficientes)[:10]

print('Termos mais associados a ALTO RISCO:')
for idx in top_alto:
    print(f'  {feature_names[idx]:<35} coef: {coeficientes[idx]:.3f}')

print('\nTermos mais associados a BAIXO RISCO:')
for idx in top_baixo:
    print(f'  {feature_names[idx]:<35} coef: {coeficientes[idx]:.3f}')

print('\nObservação:')
print('Se os termos de alto risco forem dominados por sintomas masculinos clássicos')
print('("dor no peito", "aperto", "irradiação"), isso confirma o viés de representação')
print('no vocabulário do modelo — mesmo com um dataset balanceado por gênero.')

---

## Considerações finais

Este classificador demonstra que é possível construir um sistema básico de triagem  
de risco cardiovascular com ferramentas acessíveis. As limitações são documentadas  
intencionalmente como parte da análise de governança:

- Com 30 frases de treinamento, o modelo é funcional mas não generalizável
- A análise de viés por gênero mostra que sintomas atípicos femininos podem ser subclassificados
- Nas fases seguintes, o dataset UCI Cleveland (303 pacientes) substituirá esse corpus sintético

**Este notebook não produz diagnósticos médicos. Tem finalidade exclusivamente acadêmica.**